In [1]:
# Run this each time (fast)
!pip install -U transformers accelerate pandas openpyxl

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 89.0 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3

正常生成720

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct" #Model used for text generation
cache_dir = "/content/drive/MyDrive/hf_cache"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_dir
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=cache_dir
)

model.eval()

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded.


In [8]:
# 720 samples
bank_name = "Lloyds Banking Group"
def build_prompt(row, bank_name={bank_name}):
    return f"""
You are a UK retail banking customer.

Persona:
- Age: {row['age_group']}
- Region: {row['region']}
- Income: {row['income']}
- Household: {row['household_composition']}
- Technology adoption: {row['tech_adoption']}

Question:
How likely would you be to recommend {bank_name} to a friend or colleague? Please explain your view briefly.

Task:
Provide a SHORT and CLEAR reason for your recommendation decision.Maximum 35 words.

Important guidelines:
- Base your response on general and plausible perceptions of UK retail banking.
- Do not invent specific product features or unverifiable facts.
- Do not assume the bank is inherently good or bad.
- Reflect the customer's preferences and situation based on their profile.
- Avoid repeating generic phrases unless clearly relevant.
- Do NOT contradict the persona (e.g., age vs behaviour).
  e.g.If you are young but a LATE ADOPTER / CONSERVATIVE USER: you prefer branches, human support, or simple tools. DO NOT describe yourself as elderly or old.

Output format:
1-2 sentences only.
""".strip()

In [9]:
def generate_text(prompt):
    messages = [
        {
            "role": "system",
            "content": "You are a realistic UK banking customer. Answer naturally."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return response

In [10]:
import pandas as pd

input_file = "/content/drive/MyDrive/720persona.xlsx"   # Colab path

df_personas = pd.read_excel(
    input_file,
    skiprows=2                # Skip the first two rows
)

# Clean column names (very important)
df_personas.columns = df_personas.columns.str.strip()

print(df_personas.head())
print(df_personas.columns)

   persona_id persona_code            age_group   region income  \
0           1         P001  Aged 18 to 24 years  England    low   
1           2         P002  Aged 18 to 24 years  England    low   
2           3         P003  Aged 18 to 24 years  England    low   
3           4         P004  Aged 18 to 24 years  England    low   
4           5         P005  Aged 18 to 24 years  England    low   

  household_composition                    tech_adoption  \
0   Single, no children                    Early adopter   
1   Single, no children                Selective adopter   
2   Single, no children  Late adopter/conservative users   
3   Couple, no children                    Early adopter   
4   Couple, no children                Selective adopter   

                                     persona_summary  
0  Aged 18 to 24 years | England | income=low | S...  
1  Aged 18 to 24 years | England | income=low | S...  
2  Aged 18 to 24 years | England | income=low | S...  
3  Aged 18 to 24

In [11]:
import os
import pandas as pd
import gc

output_file = "/content/drive/MyDrive/LBG_720.csv"

# If some results have already been generated, read the existing file to avoid duplicates
if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    existing_ids = set(df_existing["persona_id"].astype(str))
    print("Existing generated rows:", len(df_existing))
else:
    existing_ids = set()
    print("No existing output file found. Starting new file.")

# Skip persona_id values that have already been generated to support resuming from checkpoints
df_personas_todo = df_personas[
    ~df_personas["persona_id"].astype(str).isin(existing_ids)
].reset_index(drop=True)

print("Final personas to generate:", len(df_personas_todo))


for i, row in df_personas_todo.iterrows():
    prompt = build_prompt(row, bank_name={bank_name})

    try:
        response_text = generate_text(prompt)

        one_result = {
            "persona_id": row["persona_id"],
            "age_group": row["age_group"],
            "region": row["region"],
            "income": row["income"],
            "household_composition": row["household_composition"],
            "tech_adoption": row["tech_adoption"],
            "free_text": response_text
        }

        # Keep sample_id if it exists in the 720-persona file; otherwise no error will occur
        if "sample_id" in df_personas_todo.columns:
            one_result["sample_id"] = row["sample_id"]

        one_df = pd.DataFrame([one_result])

        if not os.path.exists(output_file):
            one_df.to_csv(output_file, index=False, encoding="utf-8-sig")
        else:
            one_df.to_csv(output_file, mode="a", header=False, index=False, encoding="utf-8-sig")

        print(f"{i+1}/{len(df_personas_todo)} done |bank: {bank_name} | persona_id: {row['persona_id']}")

        # Clear memory every 20 rows to reduce the risk of continuously increasing RAM usage
        if (i + 1) % 20 == 0:
            gc.collect()

    except Exception as e:
        print(f"Error at {i+1}, persona_id={row['persona_id']}: {e}")

No existing output file found. Starting new file.
Final personas to generate: 720
1/720 done |bank: Lloyds Banking Group | persona_id: 1
2/720 done |bank: Lloyds Banking Group | persona_id: 2
3/720 done |bank: Lloyds Banking Group | persona_id: 3
4/720 done |bank: Lloyds Banking Group | persona_id: 4
5/720 done |bank: Lloyds Banking Group | persona_id: 5
6/720 done |bank: Lloyds Banking Group | persona_id: 6
7/720 done |bank: Lloyds Banking Group | persona_id: 7
8/720 done |bank: Lloyds Banking Group | persona_id: 8
9/720 done |bank: Lloyds Banking Group | persona_id: 9
10/720 done |bank: Lloyds Banking Group | persona_id: 10
11/720 done |bank: Lloyds Banking Group | persona_id: 11
12/720 done |bank: Lloyds Banking Group | persona_id: 12
13/720 done |bank: Lloyds Banking Group | persona_id: 13
14/720 done |bank: Lloyds Banking Group | persona_id: 14
15/720 done |bank: Lloyds Banking Group | persona_id: 15
16/720 done |bank: Lloyds Banking Group | persona_id: 16
17/720 done |bank: Lloyd

In [12]:
# Check
df_free = pd.read_csv(output_file)

print("Saved rows:", len(df_free))
df_free.head()

Saved rows: 720


,persona_id,age_group,region,income,household_composition,tech_adoption,free_text
0,1,Aged 18 to 24 years,England,low,"Single, no children",Early adopter,I am very likely to recommend 'Lloyds Banking ...
1,2,Aged 18 to 24 years,England,low,"Single, no children",Selective adopter,I'm unlikely to recommend Lloyds Banking Group...
2,3,Aged 18 to 24 years,England,low,"Single, no children",Late adopter/conservative users,I'm unlikely to recommend Lloyds Banking Group...
3,4,Aged 18 to 24 years,England,low,"Couple, no children",Early adopter,As an early adopter with limited income and ho...
4,5,Aged 18 to 24 years,England,low,"Couple, no children",Selective adopter,I'm likely to recommend Lloyds Banking Group b...
